In [25]:
from pykeen.triples import TriplesFactory
import pandas as pd
from rdflib import graph, Namespace, URIRef
from rdflib.namespace import RDF

import torch
import csv
from tqdm import tqdm # A library for a smart progress bar
from collections import defaultdict
import sys
from collections import Counter

# YAGO 4.5-10

In [ ]:
# generates the base yago4.5-10 by counting and filtering

def filter_by_entity_count(file_path, output_file, threshold=10):
    """
    Filters an n-triples file to keep lines where both the subject
    and object entities appear more than a given number of times.

    Args:
        file_path (str): The path to the n-triples file.
        threshold (int): The minimum count for an entity to be included.
    """
    entity_counts = Counter()

    # --- First Pass: Count all entities ---
    # This pass builds a frequency map of every subject and object.
    print(f"INFO: Starting first pass to count entities from '{file_path}'...")
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.split()
                if len(parts) >= 3:
                    subject = parts[0]
                    obj = parts[2]
                    entity_counts[subject] += 1
                    entity_counts[obj] += 1
    except FileNotFoundError:
        print(f"ERROR: File not found at '{file_path}'", file=sys.stderr)
        return
    except Exception as e:
        print(f"ERROR: An error occurred during the first pass: {e}", file=sys.stderr)
        return

    print(f"INFO: First pass complete. Found {len(entity_counts)} unique entities.")

    # --- Second Pass: Filter and print ---
    # This pass re-reads the file and prints lines that meet the criteria.
    print(f"INFO: Starting second pass to filter lines with entity counts > {threshold}...")

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            with open(output_file, 'w', encoding='utf-8') as of:
                for line in f:
                    parts = line.split()
                    if len(parts) >= 3:
                        subject = parts[0]
                        obj = parts[2]
                        # Check if BOTH entities are above the threshold
                        if entity_counts[subject] > threshold and entity_counts[obj] > threshold:
                            # Print the original, unmodified line to standard output
                            of.write(line)
    except Exception as e:
        print(f"ERROR: An error occurred during the second pass: {e}", file=sys.stderr)
        return

    print("INFO: Filtering complete.")

input_file = 'YAGO4.5/data/original/yago4.5_triples.nt'
output_file = 'YAGO4.5/data/original/yago4.5-10.nt'
filter_by_entity_count(input_file, output_file)

In [42]:
#extracts all rdf:type triples and removes trivial 'class'
!grep '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>' YAGO4.5/data/original/yago-facts-nolit.nt | grep -v ' <http://www.w3.org/2000/01/rdf-schema#Class>' > YAGO4.5/data/original/YAGO4.5_entity_types.nt



In [52]:
# filters to get only yago4.5-10 entities and their direct classes
with open('YAGO4.5/data/original/yago4.5-10.nt') as fp:
    entities = set()
    for line in fp.readlines():
        e1, p, e2, _ = line.split()
        entities.add(e1)
        entities.add(e2)

with open('YAGO4.5/data/original/YAGO4.5_entity_types.nt') as inFile:
    with open('YAGO4.5/data/YAGO4.5-10_entity_types.nt', 'w') as outFile:
        fullcnt = 0
        cnt = 0
        for line in inFile.readlines():
            fullcnt+=1
            ent, *rest = line.split()
            if ent in entities:
                outFile.write(line)
                cnt += 1

this was a different file

In [36]:
#first do sed 's/ \.$//' yago4.5-10.nt > yago4.5-10.txt
#todo: rerun this
tf = TriplesFactory.from_path('YAGO4.5/data/original/yago4.5-10.txt', create_inverse_triples=False, load_triples_kwargs={'delimiter': ' '})
training, testing, validation = tf.split([.99, .005, .005],random_state=42)
print(training.num_triples)
print(testing.num_triples)
print(validation.num_triples)

3222052
16273
16273


In [27]:
e_conversion_dict = {value: key for key, value in training.entity_to_id.items()}
r_conversion_dict = {value: key for key, value in training.relation_to_id.items()}

In [38]:
for output_filename, input_triples in [('YAGO4.5/data/YAGO4-5-10_train.tsv',training.mapped_triples),
                                       ('YAGO4.5/data/YAGO4-5-10_valid.tsv',validation.mapped_triples),
                                       ('YAGO4.5/data/YAGO4-5-10_test.tsv',testing.mapped_triples)]:
    batch_size = 100_000
    with open(output_filename, 'w', newline='', encoding='utf-8') as f:
    # Create a CSV writer with a tab delimiter
        writer = csv.writer(f, delimiter='\t')

        # Use tqdm for a helpful progress bar
        # We iterate through the tensor in steps of batch_size
        for i in tqdm(range(0, input_triples.shape[0], batch_size)):
            # Get a chunk of the tensor
            chunk = input_triples[i : i + batch_size]

            # Move chunk to CPU (if it's on GPU) and convert to a Python list
            # .tolist() is efficient for converting a small chunk
            chunk_list = chunk.cpu().tolist()

            # Map the integer values to strings using the dictionary.
            # We use .get() for safety in case a key is missing.
            mapped_rows = [
                [e_conversion_dict.get(row[0], 'KEY_NOT_FOUND'),
                 r_conversion_dict.get(row[1], 'KEY_NOT_FOUND'),
                 e_conversion_dict.get(row[2], 'KEY_NOT_FOUND')]
                for row in chunk_list
            ]

            # Write the mapped rows to the TSV file
            writer.writerows(mapped_rows)

100%|██████████| 1/1 [00:00<00:00, 21.80it/s]


# NELL995 splits

In [57]:
default_ns = 'http://ste-lod-crew.fr/nell/ontology/'
ns = Namespace(default_ns)
g = graph.Graph()

with open('NELL995/data/original/NELLKG0.txt') as inFile:
    with open('NELL995/data/NELL995_full_graph.tsv', 'w') as outFile:
        for line in inFile:
            line=  line.replace('__','_')
            s,p,o = line.split()
            sClass, sName = s.split('_',1)
            oClass, oName = o.split('_',1)

            sClass = default_ns+sClass
            sName = default_ns+ s
            oClass = default_ns+oClass
            oName = default_ns+ o
            pName = default_ns+p

            #outFile.write(sClass+'_'+sName + '\t' + pName + '\t' + oClass+'_'+oName + '\n')
            outFile.write(default_ns + s + '\t' +default_ns + p + '\t' +default_ns + o + '\n')

            g.add((URIRef(sName), RDF.type, URIRef(sClass)))
            g.add((URIRef(oName), RDF.type, URIRef(oClass)))
            g.add((URIRef(sName), URIRef(pName), URIRef(oName)))

    g.serialize('../datasets/NELL995/data/NELL995_full_graph.nt', format='ntriples')



/Users/thezamp/miniconda3/envs/calibration/lib/python3.10/site-packages/rdflib/plugins/serializers/nt.py:41: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


In [58]:
tf = TriplesFactory.from_path('NELL995/data/NELL995_full_graph.tsv')
training, testing, validation = tf.split([.8, .1, .1],random_state=42)

In [59]:
pd.DataFrame(training.label_triples(training.mapped_triples)).to_csv('NELL995/data/NELL995_train.tsv', header=False, index=False, sep ='\t')
pd.DataFrame(testing.label_triples(testing.mapped_triples)).to_csv('NELL995/data/NELL995_test.tsv', header=False, index=False, sep ='\t')
pd.DataFrame(validation.label_triples(validation.mapped_triples)).to_csv('NELL995/data/NELL995_valid.tsv', header=False, index=False, sep ='\t')



In [60]:
types_dict = defaultdict(set)
nell = pd.read_csv('NELL995/data/NELL995_full_graph.tsv', sep='\t', header=None, names = ['s','p','o'])
for i,r in nell.iterrows():
    subject = r.s
    subject_type = subject.split('_')[0]
    obj = r.o
    object_type = obj.split('_')[0]
    types_dict[subject].add(subject_type)
    types_dict[obj].add(object_type)

with open('NELL995/data/NELL995_entity_types.nt', 'w') as f:
    for subject, types_set in types_dict.items():
        for t in types_set:
            f.write('<'+subject+'>' + '\t<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>\t<' + t + '>\t.\n')